# 1️⃣ Processamento de Dados Geoespaciais — SINOP Agro-GIS

Neste notebook carregamos e exploramos os dados geoespaciais do projeto:
- Limites municipais (IBGE)
- Preparamos estrutura para MapBiomas
- Filtramos dados específicos de Sinop-MT

## 📦 Importar Bibliotecas

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Configurar estilo
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Bibliotecas carregadas com sucesso!")
print(f"GeoPandas: {gpd.__version__}")
print(f"Pandas: {pd.__version__}")

## 🗂️ Definir Caminhos

In [ ]:
# Raiz do projeto
ROOT = Path.cwd().parent  # Suba um nível da pasta notebooks/
DATA_DIR = ROOT / "data"
LIMITE_DIR = DATA_DIR / "limite_municipal"
MAPBIOMAS_DIR = DATA_DIR / "uso_solo_mapbiomas"
MALHA_VIARIA_DIR = DATA_DIR / "malha_viaria"

print("📁 Caminhos definidos:")
print(f"  Raiz: {ROOT}")
print(f"  Dados: {DATA_DIR}")
print(f"  Limites: {LIMITE_DIR}")
print(f"  MapBiomas: {MAPBIOMAS_DIR}")
print(f"  Malha Viária: {MALHA_VIARIA_DIR}")

# Verificar se shapefiles existem
shp_files = list(LIMITE_DIR.glob("*.shp"))
if shp_files:
    print(f"\n✅ Shapefiles encontrados: {len(shp_files)}")
    for shp in shp_files:
        print(f"  - {shp.name}")
else:
    print(f"\n⚠️  Nenhum shapefile encontrado em {LIMITE_DIR}")

## 📍 Carregar Limites Municipais (IBGE 2022)

In [ ]:
# Carregar shapefile de limites municipais
shp_file = LIMITE_DIR / "MT_Municipios_2022.shp"

if shp_file.exists():
    gdf_municipios = gpd.read_file(shp_file)
    print(f"✅ Shapefile carregado: {shp_file.name}")
    print(f"\n📊 Informações básicas:")
    print(f"  Total de municípios: {len(gdf_municipios)}")
    print(f"  CRS: {gdf_municipios.crs}")
    print(f"  Colunas: {list(gdf_municipios.columns)}")
    print(f"\n🗺️ Primeiros registros:")
    gdf_municipios.head()
else:
    print(f"❌ Arquivo não encontrado: {shp_file}")

## 🔍 Filtrar Sinop (Código IBGE: 5107909)

In [ ]:
# Código IBGE de Sinop
SINOP_CODIGO_IBGE = 5107909

# Procurar pela coluna de código IBGE (pode ter nome variável)
print("📋 Colunas disponíveis:")
for col in gdf_municipios.columns:
    print(f"  - {col}: {gdf_municipios[col].dtype}")

# Encontrar a coluna de ID
id_cols = [col for col in gdf_municipios.columns 
            if 'id' in col.lower() or 'cod' in col.lower() or 'ibge' in col.lower()]
print(f"\n🔎 Colunas de ID encontradas: {id_cols}")

# Se houver coluna de ID, usá-la para filtrar
if id_cols:
    id_col = id_cols[0]
    sinop = gdf_municipios[gdf_municipios[id_col] == SINOP_CODIGO_IBGE]
    if len(sinop) > 0:
        print(f"\n✅ Sinop encontrado!")
        print(f"  Nome: {sinop['NM_MUN'].values[0]}")
        print(f"  UF: {sinop['SIGLA_UF'].values[0]}")
        print(f"  Área: {sinop.geometry.area.values[0] / 1e6:.2f} km²")
    else:
        print(f"\n⚠️  Sinop não encontrado com código {SINOP_CODIGO_IBGE}")
        print(f"\nAmostra de dados:")
        print(gdf_municipios[[id_col, 'NM_MUN', 'SIGLA_UF']].head(10))

## 📊 Estatísticas Gerais de Mato Grosso

In [ ]:
# Filtrar apenas Mato Grosso
gdf_mt = gdf_municipios[gdf_municipios['SIGLA_UF'] == 'MT']

print(f"📊 Mato Grosso:")
print(f"  Total de municípios: {len(gdf_mt)}")
print(f"  Área total: {gdf_mt.geometry.area.sum() / 1e6:.0f} km²")
print(f"  Município maior: {gdf_mt.loc[gdf_mt.geometry.area.idxmax(), 'NM_MUN']}")
print(f"  Município menor: {gdf_mt.loc[gdf_mt.geometry.area.idxmin(), 'NM_MUN']}")

# Top 10 maiores municípios
print(f"\n🏆 Top 10 maiores municípios:")
top10 = gdf_mt.nlargest(10, 'geometry')[['NM_MUN', 'geometry']].copy()
top10['Area_km2'] = top10.geometry.area / 1e6
print(top10[['NM_MUN', 'Area_km2']].to_string())

## 🗺️ Visualizar Sinop no Mapa de MT

In [ ]:
# Criar figura com Mato Grosso e Sinop destacado
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Desenhar todos os municípios de MT
gdf_mt.plot(ax=ax, color='lightblue', edgecolor='gray', alpha=0.5)

# Destaque: Sinop
sinop = gdf_municipios[gdf_municipios['CD_MUN'] == SINOP_CODIGO_IBGE]
if len(sinop) > 0:
    sinop.plot(ax=ax, color='red', alpha=0.8, edgecolor='darkred', linewidth=2)
    
    # Adicionar label
    centroid = sinop.geometry.centroid.values[0]
    ax.text(centroid.x, centroid.y, 'SINOP', fontsize=14, fontweight='bold',
            ha='center', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

ax.set_title('Mato Grosso — Localização de Sinop (2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

print("✅ Mapa gerado!")

## ⏳ Próximas Etapas

### Quando os dados abaixo forem baixados:
1. **MapBiomas** → Carregar rasters GeoTIFF
2. **Malha Viária** → Integrar rodovias/ferrovias
3. **Produção Agrícola** → Carregar CSV IBGE/CONAB

### Próximos notebooks:
- `02_analise_espacial.ipynb` — Operações geométricas e buffers
- `03_visualizacao.ipynb` — Mapas temáticos e Folium interativo